# GK-2A 12:00~14:00 Multi-year Pipeline
2019~2025년 각 8/24~8/30에 대해 12:00~14:00 KST 10분 간격 GK-2A 16채널을 수집하고, 14:00 ASOS TA/HM 라벨과 결합합니다. 마지막에 LSTM/GRU용 long CSV와 HGB/CatBoost/LightGBM용 wide(tabular) CSV를 모두 생성합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
START_MMDD, END_MMDD = '08-24', '08-30'
START_TIME, END_TIME = '12:00', '14:00'
STEP_MINUTES = 10
BRANCH = 'agent/shortterm-12to14-pipeline'
REPO_DIR = Path('/content/SME_DATA')
OUTPUT_ROOT = Path('/content/drive/MyDrive/SME_DATA/processed_station_features/shortterm_12to14_data')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('YEARS:', YEARS, flush=True)
print('OUTPUT:', OUTPUT_ROOT, flush=True)


In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
if not REPO_DIR.exists():
    subprocess.run(['git','clone','-b',BRANCH,REPO_URL,str(REPO_DIR)], check=True)
else:
    os.chdir(REPO_DIR)
    subprocess.run(['git','fetch','origin'], check=True)
    subprocess.run(['git','checkout',BRANCH], check=True)
    subprocess.run(['git','pull','origin',BRANCH], check=True)
os.chdir(REPO_DIR)
!pip -q install -e .
!pip -q install requests pandas numpy xarray h5netcdf netCDF4 pyyaml
print('GitHub/패키지 준비 완료', flush=True)


In [ ]:
import os
from getpass import getpass
try:
    from google.colab import userdata
    key = userdata.get('KMA_API_KEY')
except Exception:
    key = None
if not key:
    key = getpass('KMA_API_KEY 입력: ').strip()
if not key:
    raise ValueError('KMA_API_KEY가 비어 있습니다.')
os.environ['KMA_API_KEY'] = key
print('KMA_API_KEY 설정 완료', flush=True)


## 실행 전 진단
이 셀에서 모든 항목이 정상이어야 합니다. 여기서도 출력이 안 나오면 Colab 런타임 자체가 이전 프로세스에 묶인 상태이므로 런타임을 재시작한 뒤 위 셀부터 다시 실행하세요.


In [ ]:
import os, sys
print('=== PRE-FLIGHT CHECK ===', flush=True)
print('Python:', sys.executable, flush=True)
print('REPO_DIR exists:', REPO_DIR.exists(), REPO_DIR, flush=True)
print('multi-year script exists:', (REPO_DIR/'scripts/collect_shortterm_years.py').exists(), flush=True)
print('collector exists:', (REPO_DIR/'scripts/collect_shortterm_12to14.py').exists(), flush=True)
print('builder exists:', (REPO_DIR/'scripts/build_shortterm_12to14_dataset.py').exists(), flush=True)
print('station_list exists:', (REPO_DIR/'data/metadata/station_list.csv').exists(), flush=True)
print('Drive output exists:', OUTPUT_ROOT.exists(), flush=True)
print('KMA_API_KEY set:', bool(os.environ.get('KMA_API_KEY')), flush=True)
print('=== CHECK END ===', flush=True)
assert REPO_DIR.exists()
assert (REPO_DIR/'scripts/collect_shortterm_years.py').exists()
assert (REPO_DIR/'scripts/collect_shortterm_12to14.py').exists()
assert (REPO_DIR/'scripts/build_shortterm_12to14_dataset.py').exists()
assert (REPO_DIR/'data/metadata/station_list.csv').exists()
assert os.environ.get('KMA_API_KEY')


## Multi-year 수집 + tabular 변환 + TA/HM 병합
실시간 로그 스트리밍 방식입니다. 실행 즉시 `[COLAB] process starting...`이 보여야 정상입니다.


In [ ]:
import os, sys, subprocess
cmd = [
    sys.executable, '-u', 'scripts/collect_shortterm_years.py',
    '--years', *[str(y) for y in YEARS],
    '--start-mmdd', START_MMDD, '--end-mmdd', END_MMDD,
    '--start-time', START_TIME, '--end-time', END_TIME,
    '--step-minutes', str(STEP_MINUTES),
    '--output-root', str(OUTPUT_ROOT),
]
print('[COLAB] process starting...', flush=True)
print('[COLAB] cwd =', REPO_DIR, flush=True)
print('[COLAB] command =', ' '.join(cmd), flush=True)
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
proc = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
returncode = proc.wait()
print(f'\n[COLAB] process finished: returncode={returncode}', flush=True)
if returncode != 0:
    raise RuntimeError(f'multi-year pipeline failed: returncode={returncode}')


## 결과 검수


In [ ]:
import pandas as pd
from pandas.errors import EmptyDataError
COMBINED_DIR = OUTPUT_ROOT / 'datasets' / 'combined'
SUMMARY = COMBINED_DIR / 'shortterm_multiyear_summary.csv'
LONG = COMBINED_DIR / 'shortterm_long_2019to2025.csv'
WIDE = COMBINED_DIR / 'shortterm_wide_2019to2025.csv'
LABELS = COMBINED_DIR / 'shortterm_labels_1400_2019to2025.csv'
FAIL = COMBINED_DIR / 'shortterm_collection_failures_2019to2025.csv'
BUILD_MISSING = COMBINED_DIR / 'shortterm_build_missing_2019to2025.csv'
summary = pd.read_csv(SUMMARY)
display(summary)
long_df = pd.read_csv(LONG)
wide_df = pd.read_csv(WIDE)
labels_df = pd.read_csv(LABELS)
print('LONG:', long_df.shape, 'WIDE:', wide_df.shape, 'LABELS:', labels_df.shape)
print('Years:', sorted(wide_df['Year'].unique().tolist()))
print('Stations:', wide_df['STN_ID'].nunique())
print('Long timesteps:', sorted(long_df['TimeKST'].unique().tolist()))
print('TA missing:', labels_df['TA'].isna().sum(), 'HM missing:', labels_df['HM'].isna().sum())
for path, name in [(FAIL,'collection failures'), (BUILD_MISSING,'build issues')]:
    try:
        df = pd.read_csv(path) if path.exists() and path.stat().st_size else pd.DataFrame()
    except EmptyDataError:
        df = pd.DataFrame()
    print(name + ':', len(df))
